# This project explains how a plan of building a data center in a small town north of NYC fell apart, centering on power use and public sentiment.
# Why this project? something in local news cycle and happening near where I live
# Due data: Due Jul 12 by 11:59pm 
# After many trial-and-errors, which is saved in a seperate folder called "data center", I've decided to keep this project real simple and start over. I'm focusing on power use at a Hudson Valley town where a data center was planned
# I'm using this notebook to get some bascic data: power use data in East Fishkill, where the data center was proposed (A. local power rates increases vs NYS average; B. scale 1 GW against local grid capacity)
Main sources: 
--Central Hudson rates
https://www.cenhud.com/globalassets/pdf/my-energy/dg/average-rates.pdf
--2026 rate hike
https://www.cenhud.com/globalassets/pdf/account-resources/rates/2025-2028-rate-plan-brochure.pdf
--NY state residential power rates for comparison (**eventually dropped comparison because i'm not 100% sure it's apples-to-apples)
https://www.nyserda.ny.gov/Energy-Prices/Electricity/Monthly-Avg-Electricity-Residential
--backlash reports
https://highlandscurrent.org/2026/07/03/data-centers-face-backlash/
--*developer's request with NYISO*
https://www.nyiso.com/documents/20142/56802634/03d_Q1738_SIS%20Scope_2-3-26-TPAS.pdf/23c43711-119f-f819-0f62-0d7775ff3eb6
# A seperate text data, to showcase the main complains/concerns of local residents, will be addressed in a seperate notbook (because scraping the Youtbue transcripts of public hearings turned out more difficult than I thought)
Main source: 
https://www.youtube.com/user/EF22NY
# For visualization, perhaps some sort of simple map to show location (google earth image comparison???), charts to show rate increases. IF I manage to scrape the transcrips, I may think of a way to reflect the key concerns from local residents (power, job, environment?) (**I didn't end up using a map to avoid overcrowding the story with images and being distractive to readers. Hopefully next project!!)

## to tell the story, I eventually decided the structure as below:
 --tension: the word cloud and transcripts capture a community deeply worried about infrastructure, water, and the grid.
 --context: Your line chart proves that local rates are already on a volatile upward trajectory.

In [1]:
# local power rates are from this pdf report by Central Hudson, the power company: https://www.cenhud.com/globalassets/pdf/my-energy/dg/average-rates.pdf

import pandas as pd


# 1. Create the base dataset (Fixed the missing years list here)
data = {
    "Year": [2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016],
    "Rate_per_kWh": [0.2436, 0.2201, 0.2135, 0.2202, 0.1635, 0.1472, 0.1385, 0.1541, 0.1310, 0.1308]
}

# Load the data into a Pandas DataFrame
power_df = pd.DataFrame(data)

# 2. Define the benchmark usage variable (600 kWh) (same pdf source: Average SC1 Residential Rates (based on 600 kWh) 
monthly_usage_kwh = 600

# 3. Calculate the columns using math operations
power_df["Estimated Monthly Bill"] = power_df["Rate_per_kWh"] * monthly_usage_kwh
power_df["Estimated Annual Power Cost"] = power_df["Estimated Monthly Bill"] * 12

# 4. Clean up the formatting for presentation
power_df["Rate_per_kWh"] = power_df["Rate_per_kWh"].map("${:.4f}".format)
power_df["Estimated Monthly Bill"] = power_df["Estimated Monthly Bill"].map("${:.2f}".format)
power_df["Estimated Annual Power Cost"] = power_df["Estimated Annual Power Cost"].map("${:.2f}".format)

# Display the final cleaned table
power_df


,Year,Rate_per_kWh,Estimated Monthly Bill,Estimated Annual Power Cost
0,2025,$0.2436,$146.16,$1753.92
1,2024,$0.2201,$132.06,$1584.72
2,2023,$0.2135,$128.10,$1537.20
3,2022,$0.2202,$132.12,$1585.44
4,2021,$0.1635,$98.10,$1177.20
5,2020,$0.1472,$88.32,$1059.84
6,2019,$0.1385,$83.10,$997.20
7,2018,$0.1541,$92.46,$1109.52
8,2017,$0.1310,$78.60,$943.20
9,2016,$0.1308,$78.48,$941.76


In [2]:
power_df.to_csv("power_rates.csv", index=False)

# **Dropped comparison between East Fishkill power rate and NYS mean/average because I am not 100% sure the calculated rates are comparable
data sources: https://www.nyserda.ny.gov/Energy-Prices/Electricity/Monthly-Avg-Electricity-Residential

In [5]:

import altair as alt
import pandas as pd

## 1. load data needed

df_rates = pd.read_csv("power_rates.csv")

# Clean dollar values
df_rates["Estimated Monthly Bill"] = (
    df_rates["Estimated Monthly Bill"]
    .str.replace("$", "", regex=False)
    .astype(float)
)

# Make sure Year is numeric
df_rates["Year"] = df_rates["Year"].astype(int)

## 2. reshape data for Altair

df_long = df_rates.melt(
    id_vars=["Year"],
    value_vars=["Estimated Monthly Bill"],
    var_name="Location",
    value_name="Bill",
)

df_long["Location"] = "Central Hudson (East Fishkill)"

## 3. main chart line

line = (
    alt.Chart(df_long)
    .mark_line(point=True)
    .encode(
        x=alt.X(
            "Year:Q",
            title="Year",
            axis=alt.Axis(
                values=list(range(2016, 2026)),
                format="d",
                labelAngle=0
            )
        ),
        y=alt.Y(
            "Bill:Q",
            title="Estimated Monthly Bill ($)"
        ),
        color=alt.Color(
            "Location:N",
            legend=None,
            scale=alt.Scale(range=["#F11410"])
        )
    )
)

## vertical annotation line for 2022

annotation_df = pd.DataFrame({
    "Year": [2022]
})

rule = (
    alt.Chart(annotation_df)
    .mark_rule(
        color="gray",
        strokeDash=[4,4],
        size=1.5
    )
    .encode(
        x="Year:Q"
    )
)

## 5. annotation text //needed to be centered over the 2022 vertical line and inside the plotting area//

# - Center the text over the 2022 vertical line.
# - Keep the annotation INSIDE the plotting area.
# - Split into two lines for readability.

annotation = (
    alt.Chart(annotation_df)
    .mark_text(
        text="Rate jump amid\nenergy supply shocks",
        align="center",      # Center text on the vertical rule
        baseline="top",
        dy=8,                # Small offset from the top of the plot
        fontSize=12,
        fontWeight="bold",
        color="black"
    )
    .encode(
        x=alt.X("Year:Q"),   # Same x-position as the dotted rule (2022)
        y=alt.value(10)      # Place near the top INSIDE the chart
    )
)
## combine layers and add //a real title//, width, height, and styling

chart = (
    (line + rule + annotation)
    .properties(
        title="Monthly power bills have climbed steadily, with a sharp jump in 2022",
        width=600,
        height=350,
    )
    .configure_title(
        fontSize=16,
        anchor="middle",
        font="sans-serif"
    )
    .configure_axis(
        grid=True,
        gridDash=[2,2],
        labelFont="sans-serif"
    )
)


chart.show()


alt.LayerChart(...)

At full capacity, the proposed 1,000 MW East Fishkill facility is projected to consume 8.76 billion kWh annually. Based on standard federal energy metrics for New York households, this matches the power footprint of approximately 1.2 million homes under standard operating conditions. However, because data centers and residential consumption habits change, this scale operates across three specific scenarios:

In [10]:
import pandas as pd

# 1. Define the project parameters (The raw data from the NYISO filing)
project_name = "1 Gig Data Center East Fishkill"
data_center_capacity_mw = 1000  # 1,000 MW

# Calculate the maximum annual kWh a 1,000 MW facility can pull (Capacity * Hours * Days * 1000 to get kW)
max_annual_dc_kwh = data_center_capacity_mw * 24 * 365 * 1000

# 2. Build the data journalism scenario matrix
scenarios = {
    "Scenario Name": [
        "The Operational Baseline", 
        "The Downtime Shift", 
        "The Peak Heatwave Shift"
    ],
    "Data Center Capacity Factor": [1.00, 0.85, 1.00],        # 100%, 85%, 100%
    "Avg Monthly Home Usage (kWh)": [571, 571, 800]           # Standard vs. Peak Summer
}

# Load the matrix into a Pandas DataFrame
df_scenarios = pd.DataFrame(scenarios)

# 3. Perform the automated math operations across the columns
# Total annual energy consumed by the data center in this scenario
df_scenarios["Data Center Annual kWh"] = max_annual_dc_kwh * df_scenarios["Data Center Capacity Factor"]

# Total annual energy consumed by a single household in this scenario
df_scenarios["Home Annual kWh"] = df_scenarios["Avg Monthly Home Usage (kWh)"] * 12

# The core calculation: Data Center total divided by Home total
df_scenarios["Equivalent NY Homes"] = df_scenarios["Data Center Annual kWh"] / df_scenarios["Home Annual kWh"]

# 4. Clean up the formatting for your editing team (Rounding to integers)
df_scenarios["Equivalent NY Homes"] = df_scenarios["Equivalent NY Homes"].round(0).astype(int)

# Display your automated scenario results table
df_scenarios[["Scenario Name", "Data Center Capacity Factor", "Avg Monthly Home Usage (kWh)", "Equivalent NY Homes"]]


,Scenario Name,Data Center Capacity Factor,Avg Monthly Home Usage (kWh),Equivalent NY Homes
0,The Operational Baseline,1.00,571,1278459
1,The Downtime Shift,0.85,571,1086690
2,The Peak Heatwave Shift,1.00,800,912500


In [13]:
df_scenarios.to_csv ("power_use_senarios.csv", index=False)

# for visualization perhaps simple line chart for the power rates over the years, and somehow a SVG (Scalable Vector Graphics) for the equivalent NY homes in different scenarios.
# I'm not going to overcrowd the story with too many graphic elements, but I do want to show the relative scale of the data center's power consumption compared to a typical NY household. (I tried that in Code Pen) https://codepen.io/editor/Liyan-Qi/pen/019f4842-0429-7bf8-a094-566490d86b10


In [16]:
#The mentor mentioned a quick frequency check of the topic, which is a good idea.
#Again, I'm only going to mention one or two setences in the text, not thinking about generating another image. 

import pandas as pd

df = pd.read_csv("town_board_transcripts.csv")

for _, row in df.iterrows():
    text = row["Full Text"]
    count = text.count("data center") + text.count("data centers")
    print(f"{row['Month']}: {count} mentions")


April: 0 mentions
May: 5 mentions
May: 30 mentions
June: 195 mentions
